<a href="https://colab.research.google.com/github/KCL-Health-NLP/nlp_examples/blob/master/ann/fine_tuning_with_huggingface-answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning a Hugging Face model - answers

**This is the "answers" version of this notebook. It differs from the "question" version in that the exercise to do model training and evaluation has been completed your model.**

Based on this [Hugging Face blog](https://huggingface.co/blog/sentiment-analysis-python)



## Fine tuning

In the last practical, we trained a new transformer model on a classification task, using a supervised method.

In fine-tuning, we take an existing language model that has been pre-trained on some task, and train that model for some other, usually related, task. Pre-training followed by fine-tuning is a kind of transfer learning - learning knowledge from one task, and applying it to another.

Often, pre-training will:
* be unsupervised, e.g. predicting masked words in a corpus of sentences
* include large amounts of training data, to ensure generalisability
* learn a large number of parameters, to ensure a rich representation

We will fine tune the popular pre-trained transformer [BERT Base](https://aclanthology.org/N19-1423.pdf) model. The original BERT Base model was trained on 11000 books and the whole of wikipedia, and has 110 million parameters.

BERT base was trained on two tasks:
* a masked language task - given a sequence with masked words, predict the missing masked words
* given two sentences, predict whether they follow on from each other in the original data

It is especially useful in tasks that use long sequences of data, e.g. text classification. The practical will fine tune BERT base to classify IMDb movie reviews, using the [Hugging Face transformers library](https://huggingface.co/docs/transformers/index).

The notebook steps through the basics of setting up fine-tuning, and then asks you to complete model training and evaluation.

## Hugging Face

[Hugging Face](https://huggingface.co/) is company that provides:

* a very popular machine learning library
* a repository for sharing trained models and datasets

The Hugging Face library is especially noted for its transformer support. It has a very high level API, with just a few simple lines of code needed to perform many deep learning tasks. By default, it uses the [PyTorch tensor library](https://pytorch.org/), but there is also support for Keras / TensorFlow and for other tensor libraries. In this practical we will use the Hugging Face API.

We will load the pre-trained BERT Base model from Hugging Face. We will use a version trained on case sensitive text (i.e. it includes both capital and lower case characters. It is worth reading the full model description on the [BERT Base cased model card](https://huggingface.co/bert-base-cased) in the Hugging Face repository.


## Using with GPUs

The execution time of this code will benefit from the use of GPUs. To select a GPU runtime in colab:

* Select the *Runtime* menu
* Select the *Change runtime type* submenu
* In the dialog that appears, under *Hardware accelerator* select *GPU*
* Your existing runtime will disconnect, and you will be allocated and connected to a new GPU runtime.

## Install packages
We need to install several Hugging Face libraries, which are not provided by default in Colab.

In [ ]:
# Install Hugging Face transformers, evaluate and datasets libraries
!pip install transformers evaluate

# -U : make sure we have the lates datasets package, to avoid a bug
!pip install -U datasets

## ***Restart your runtime***
**You need to restart your runtime in order for the above packages to be made available for imports**

* Menus
  * Runtime
    * Restart session

## Imports

In [ ]:
# numpy as usual...
import numpy as np

# Hugging Face datasets library has facilities for
# loading datasets direct from the Hugging Face
# datatsets repository.
from datasets import load_dataset

# The Hugging Face sequence classifier - this is what we will
# load BERT Base in to, and which we will fine-tune
from transformers import AutoModelForSequenceClassification

# Many pre-trained transformer models have their own
# specific tokenisation. There is also a class to assist with
# padding, and to collate data for faster training
from transformers import AutoTokenizer, DataCollatorWithPadding

# Hugging Face has special classes to hold training arguments
# and to train the model
from transformers import TrainingArguments, Trainer

# We can use a Hugging Face pipeline to construct an end-to-end
# classifier from a model, that takes text as input and
# outputs the class
from transformers import pipeline

# The Hugging Face evaluation package
import evaluate

## Load the data

As in the previous notebooks, we will use the IMDb review dataset. Hugging Face has a repository of popular datasets, from which wew will load IMDb. In this, movie reviews are labels as either positive sentiment (label 1), or negative sentiment (label 0).

A Hugging Face DatasetDict (dataset dictionary) object can hold multiple subsets (e.g. training, testing), and has methods for accessing different portions, splitting, shuffling, etc.

Take a look at the object created.

In [ ]:
# Load the dataset
ds_imdb = load_dataset("imdb")

# Let's take a look at it
ds_imdb

## Reduce size of dataset to speed up

Transformers can be slow to train, even when fine-tuning for simple tasks. For the sake of time, We will therefore fine-tune on a portion of IMDb. In a real setting you would want to use as much data as you can. Run the code below and make sure you understand:

* the structure of the DatasetDict
* the size of this new DatasetDict and it's subsets
* the difference between the test subset in this new dataset, and the test subset in the above dataset

In [ ]:
# Make a small training dataset from the 'train' part of IMDb.
# Shuffle the dataset using a random seed, and then select the
# first 600 reviews.
ds_train_sm = ds_imdb['train'].shuffle(seed=42).select(range(600))

# Create a train / test split.
ds_train_sm = ds_train_sm.train_test_split(test_size=0.2)

# Let's take a look at the structure of
ds_train_sm

And let's look at a single example from one of the datasets:

In [ ]:
ds_imdb['train'][5]

## Tokenise

The BERT models have been built with their own specific method of tokenisation, WordPiece tokenisation. This means that we have to tokenise our text with exactly the same method.

This is trained by starting with a vocabulary of tokens consisting of every character in the dataset, and then iteratively merging frequent combinations of these character tokens. The end result is tokenisation that splits words in to fragments, or pieces. It is especially useful when dealing with unseen words. Hugging Face has a good [explanation of WordPiece tokenisation](https://huggingface.co/learn/nlp-course/chapter6/6?fw=pt).

We start by loading the BERT Base cased pretrained tokenisation model form Hugging Face. Hugging Face has a tokenizer class that will detect the type of tokenizer to build from the model being loaded.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

Let's try it out on some text.

In [ ]:
encoding = tokenizer.encode('the quick brown fox jumped over the lazy dog')
print(encoding)

We had 9 words, but we get 11 integers. Why? Let's convert them back to see what tokens they represent.

In [ ]:
print(tokenizer.convert_ids_to_tokens([101, 1103, 3613, 3058, 17594, 4874, 1166, 1103, 16688, 3676, 102]))

Two extra tokens have been added, ```[CLS]``` to represent the class of a sequence, and ```[SEP]``` to separate this sequence from some output sequence.

Let's look at some others:

In [ ]:
print(tokenizer.convert_ids_to_tokens([101,102,103,904,905,2006,2007,3008,'12807']))

We have a mix of special tokens, single characters that have never been merged, whole words and part words.

What is the [MASK] token for?

What happens if we try to tokenise some made up words?

In [ ]:
encoding = tokenizer.encode('elephere')
print(encoding)
encoding = tokenizer.encode('protoshere')
print(encoding)

Can you make out the individual tokens in the integer? Try converting them back to tokens:

In [ ]:
print(tokenizer.convert_ids_to_tokens([101, 8468, 8043, 12807, 102]))

Now we will define a simple tokenisation function that takes a batch of text and labels as input, takes out the text part of it, and returns the tokenised text and the labels. We will also define a DataCollator to add padding.

In [ ]:
# Tokenise a batch
def tokenize(batch):
   return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

# Add padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Now we can tokenize our training and validation sets, using the Dataset map method, which applies a function to every row.

We will print out an example. Take a look - can you see what the tokenisation has done, in addition to creating token vectors?

* attention mask - used to mask / select which tokens to consider, e.g. if we have padded with zeros, we might want to ignore our zero tokens
* token type ids - used to mask / select input and output sequences, if we have both. 0 selects input, 1 selects output.

In [ ]:
# Tokenise our datasets and print some out

tokenized_train = ds_train_sm['train'].map(tokenize, batched=True)
tokenized_test = ds_train_sm['test'].map(tokenize, batched=True)
tokenized_train[5]

## Create the model

We will now create our model. We will use a ```AutoModelForSequenceClassification``` which creates a PyTorch sequence classification model.

Like the tokeniser, this is a [Hugging Face AutoModel](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html) which will load a named pre-trained model from the Hugging Face repository, and detect exactly what class to instantiate. As we are loading a BERT model, it will create a transformer.

In [ ]:
# Load and compile our model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased")

## Set up evaluation

We will want to evaluate to evaluate our mode. We do this by defining an evaluation function that we can then pass to our model trainer.

In [ ]:
# function to load some evaluation metrics
# and compute them on a prediction
def compute_metrics(eval_pred):

   # load the metrics we want to use
   eval_accuracy = evaluate.load("accuracy")
   eval_f1 = evaluate.load("f1")

   # get the predicitons and reference labels from
   # the data
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)

   # calculate and return the metrics
   accuracy = eval_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
   f1 = eval_f1.compute(predictions=predictions, references=labels)["f1"]
   return {"accuracy": accuracy, "f1": f1}

## Set up training

Now we need to get ready for training. Because there can be so many training arguments and hyperparameters, Hugging Face splits these in to a separate TrainingArguments class, and then passes these to a Trainer class as a parameter.

We will set up just a few arguments. There are many more, as described in the [TrainingArguments documentation](https://huggingface.co/docs/transformers/v4.52.3/en/main_classes/trainer#transformers.TrainingArguments).

We have set the push_to_hub argument to False. Take a look in the documentation to see what this would do if we set it to True.

In [ ]:
# This is where models will be stored, on disk
repo_name = "finetuning-sentiment-model"

# Set up arguments - there are many more possibilities, see the
# documentation for details
training_args = TrainingArguments(
   output_dir=repo_name,
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=2,
   weight_decay=0.01,
   save_strategy="epoch",
   push_to_hub=False,
   report_to="none"
)

# Define the trainer, using the objects and functions
# we have set up above.
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics
)

## Train the model

Finally, we can fit the model to the training data, and evaluate.

In [ ]:
trainer.train()
trainer.evaluate()

## How do we apply the model?

There are many ways to apply the model to unseen text. Here's the simplest.

In [ ]:
# Some text to classify
text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."


# LABEL_1 is positive, LABEL_0 is negative
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
classifier(text)


## Visualising with TensorBoard

In previous practicals, we have plotted simple matplotlib graphs of training loss and accuracy. There are several tools specifically for this, some integrated with Hugging Face. The below code uses Tensorflow's tool, [TensorBoard](https://huggingface.co/docs/hub/en/tensorboard) to show how you might do this.

A popular laternative is [Weights and Biases (W&B)](https://docs.wandb.ai/guides/integrations/huggingface/)


In [ ]:
# Training arguments - same as before, but with logging defined and
# the "report_to" argument set to tensorboard. We have also set to
# 10 epochs so that we can gather more data.
training_args = TrainingArguments(
   output_dir=repo_name,
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=10,
   weight_decay=0.01,
   save_strategy="epoch",
   push_to_hub=False,
   report_to="tensorboard",
   eval_strategy="steps",
   eval_steps=50,
   logging_strategy="steps",
   logging_steps=50,
)

# The trainer - as before
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics
)

In [ ]:
# Train
trainer.train()

In [ ]:
# Import, load, and display TensorBoard.
import tensorflow as tf
import datetime, os
%load_ext tensorboard
%tensorboard --logdir '{repo_name}'/runs